In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
import re

# Data visualization
import seaborn as sns
import matplotlib.pyplot as plt
# from ydata_profiling import ProfileReport
import scipy.stats as stats 
from scipy.stats import mode

# Suppress warnings
import warnings 
warnings.filterwarnings("ignore")

# Preprocessing
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.decomposition import PCA

# Model selection and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score,RandomizedSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, roc_curve, accuracy_score, auc, precision_recall_curve, average_precision_score

# Machine learning models
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
import xgboost as xgb
import lightgbm as lgb

# Visualization of feature importances
from yellowbrick.model_selection import FeatureImportances

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('float_format', '{:f}'.format)

In [ ]:
df = pd.read_csv('../data/processed/train_cleaned.csv')
print('This dataset has %d rows dan %d columns.\n' % df.shape)
df.head()

In [ ]:
X = df.drop(columns=['Credit_Score'])
y = df['Credit_Score']

In [ ]:
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_res, y_res = smote.fit_resample(X, y)

In [ ]:
print(y_res.value_counts())

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [ ]:
# scale_pos_weight = len(y_train) / (2 * sum(y_train)) 

In [ ]:
xgb_model = xgb.XGBClassifier(
    eval_metric='mlogloss', 
    tree_method='hist', 
    gpu_id=0
)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_val_scaled)

In [ ]:
accuracy = accuracy_score(y_val, y_pred_xgb)
print(f"Accuracy: {accuracy:.4f}")

# Precision
precision = precision_score(y_val, y_pred_xgb, average='binary')  # Use 'micro', 'macro', or 'weighted' for multi-class
print(f"Precision: {precision:.4f}")

# Recall
recall = recall_score(y_val, y_pred_xgb, average='binary')  # Use 'micro', 'macro', or 'weighted' for multi-class
print(f"Recall: {recall:.4f}")

# F1-Score
f1 = f1_score(y_val, y_pred_xgb, average='binary')  # Use 'micro', 'macro', or 'weighted' for multi-class
print(f"F1-Score: {f1:.4f}")

if len(np.unique(y_val)) == 2:
    y_pred_proba_xgb = xgb_model.predict_proba(X_val_scaled)[:, 1]
    roc_auc = roc_auc_score(y_val, y_pred_proba_xgb)
    print(f"ROC AUC (XGBoost): {roc_auc:.4f}")
else:
    roc_auc = roc_auc_score(y_val, xgb_model.predict_proba(X_val_scaled), multi_class='ovr', average='macro')
    print(f"ROC AUC (Multi-Class - XGBoost): {roc_auc:.4f}")

# Confusion Matrix
conf_matrix = confusion_matrix(y_val, y_pred_xgb)
print("Confusion Matrix:")
print(conf_matrix)

# Classification Report
class_report = classification_report(y_val, y_pred_xgb)
print("Classification Report:")
print(class_report)

# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y_val), yticklabels=np.unique(y_val))
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
lgb_model = lgb.LGBMClassifier()  
lgb_model.fit(X_train_scaled, y_train)
y_pred_lgb = lgb_model.predict(X_val_scaled)

In [ ]:
accuracy = accuracy_score(y_val, y_pred_lgb)
print(f"Accuracy: {accuracy:.4f}")

# Precision
precision = precision_score(y_val, y_pred_lgb, average='binary')  # Use 'micro', 'macro', or 'weighted' for multi-class
print(f"Precision: {precision:.4f}")

# Recall
recall = recall_score(y_val, y_pred_lgb, average='binary')  # Use 'micro', 'macro', or 'weighted' for multi-class
print(f"Recall: {recall:.4f}")

# F1-Score
f1 = f1_score(y_val, y_pred_lgb, average='binary')  # Use 'micro', 'macro', or 'weighted' for multi-class
print(f"F1-Score: {f1:.4f}")

# ROC AUC for binary classification
if len(np.unique(y_val)) == 2:
    y_pred_proba_lgb = lgb_model.predict_proba(X_val_scaled)[:, 1]
    roc_auc = roc_auc_score(y_val, y_pred_proba_lgb)
    print(f"ROC AUC (LightGBM): {roc_auc:.4f}")
else:
    roc_auc = roc_auc_score(y_val, lgb_model.predict_proba(X_val_scaled), multi_class='ovr', average='macro')
    print(f"ROC AUC (Multi-Class - LightGBM): {roc_auc:.4f}")

# Confusion Matrix
conf_matrix = confusion_matrix(y_val, y_pred_lgb)
print("Confusion Matrix:")
print(conf_matrix)

# Classification Report
class_report = classification_report(y_val, y_pred_lgb)
print("Classification Report:")
print(class_report)

# Plot Confusion Matrix (Optional)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y_val), yticklabels=np.unique(y_val))
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
param_grid = {
    'num_leaves': [31, 63, 127],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

grid_search = GridSearchCV(estimator=lgb.LGBMClassifier(), param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)
best_lgb_model = grid_search.best_estimator_

In [ ]:
param_grid = {
    'learning_rate': np.logspace(-4, 0, 5), 
    'n_estimators': [100, 200, 300, 400], 
    'max_depth': [3, 5, 7, 10], 
    'subsample': [0.6, 0.8, 1.0],  
    'colsample_bytree': [0.6, 0.8, 1.0], 
    'gamma': [0, 0.1, 0.2, 0.3],  
    'reg_alpha': [0, 0.1, 1, 10],  
    'reg_lambda': [0.1, 1, 10]  
}

xgb_model = xgb.XGBClassifier(
    tree_method='gpu_hist', 
    gpu_id=0,  
    use_label_encoder=False  
)

grid_search = GridSearchCV(
    xgb_model, param_grid, cv=3, n_jobs=-1, verbose=1, scoring='accuracy'
)

grid_search.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], verbose=True)

print(f"Best Hyperparameters: {grid_search.best_params_}")

y_pred = grid_search.best_estimator_.predict(X_val_scaled)

accuracy = accuracy_score(y_val, y_pred)
conf_matrix = confusion_matrix(y_val, y_pred)
classification_rep = classification_report(y_val, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Confusion Matrix: \n{conf_matrix}")
print(f"Classification Report: \n{classification_rep}")